In [1]:
# ============================================================
# PROJECT: UPI Fraud & Risk Analytics
# PURPOSE: Generate realistic synthetic dataset
# ============================================================

import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker('en_IN')   # Indian locale for realistic names/cities
np.random.seed(42)
random.seed(42)

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
N_USERS        = 5000
N_MERCHANTS    = 500
N_TRANSACTIONS = 50000
FRAUD_RATE     = 0.032   # 3.2% fraud — realistic for UPI

CITIES = ['Mumbai','Delhi','Bangalore','Hyderabad','Pune',
          'Chennai','Kolkata','Ahmedabad','Jaipur','Nagpur',
          'Surat','Lucknow','Kanpur','Bhopal','Indore']

MERCHANT_CATEGORIES = ['Food & Dining','E-Commerce','Grocery',
                       'Recharge & Bills','Travel','Healthcare',
                       'Entertainment','Fuel','Education','Wallet Transfer']

CHANNELS = ['UPI','Wallet','Debit Card','Credit Card']

KYC_STATUS = ['Full KYC','Minimal KYC','Non-KYC']

TXN_STATUS = ['Success','Failed','Pending','Reversed']

FAILURE_REASONS = ['Insufficient balance','Bank server timeout',
                   'Invalid VPA','Daily limit exceeded',
                   'Account blocked','Network error']

FRAUD_TYPES = ['Account Takeover','Phishing','Unauthorized Transfer',
               'Merchant Fraud','Velocity Fraud','SIM Swap']


In [2]:

# ─────────────────────────────────────────
# TABLE 1: USERS
# ─────────────────────────────────────────
def generate_users():
    users = []
    for i in range(N_USERS):
        account_age = np.random.randint(30, 1825)  # 1 month to 5 years
        users.append({
            'user_id'          : f'USR{str(i+1).zfill(5)}',
            'name'             : fake.name(),
            'city'             : random.choice(CITIES),
            'age_group'        : random.choice(['18-25','26-35','36-45','46-60','60+']),
            'account_age_days' : account_age,
            'kyc_status'       : np.random.choice(KYC_STATUS, p=[0.65, 0.25, 0.10]),
            'registration_date': (datetime.now() - timedelta(days=account_age)).date()
        })
    return pd.DataFrame(users)


In [3]:
# ─────────────────────────────────────────
# TABLE 2: MERCHANTS
# ─────────────────────────────────────────
def generate_merchants():
    merchants = []
    for i in range(N_MERCHANTS):
        reg_days = np.random.randint(60, 1500)
        merchants.append({
            'merchant_id'       : f'MER{str(i+1).zfill(4)}',
            'merchant_name'     : fake.company(),
            'category'          : random.choice(MERCHANT_CATEGORIES),
            'city'              : random.choice(CITIES),
            'registration_date' : (datetime.now() - timedelta(days=reg_days)).date(),
            'is_verified'       : np.random.choice([True, False], p=[0.85, 0.15])
        })
    return pd.DataFrame(merchants)


In [4]:
# ─────────────────────────────────────────
# TABLE 3: TRANSACTIONS (core table)
# ─────────────────────────────────────────
def generate_transactions(users_df, merchants_df):
    txns = []
    # Generate random timestamps over last 12 months
    start_date = datetime.now() - timedelta(days=365)
    
    user_ids     = users_df['user_id'].tolist()
    merchant_ids = merchants_df['merchant_id'].tolist()
    
    for i in range(N_TRANSACTIONS):
        rand_seconds = np.random.randint(0, 365*24*3600)
        txn_time     = start_date + timedelta(seconds=rand_seconds)
        
        # Realistic amount distribution (most txns are small)
        amount = round(np.random.choice([
            np.random.uniform(10, 500),      # 60% small txns
            np.random.uniform(500, 5000),    # 30% medium
            np.random.uniform(5000, 50000)   # 10% large
        ], p=[0.60, 0.30, 0.10]), 2)
        
        status = np.random.choice(TXN_STATUS, p=[0.88, 0.07, 0.03, 0.02])
        
        txns.append({
            'txn_id'      : f'TXN{str(i+1).zfill(7)}',
            'user_id'     : random.choice(user_ids),
            'merchant_id' : random.choice(merchant_ids),
            'amount'      : amount,
            'timestamp'   : txn_time,
            'status'      : status,
            'channel'     : np.random.choice(CHANNELS, p=[0.55, 0.20, 0.15, 0.10]),
            'device_type' : random.choice(['Mobile','Desktop','Tablet']),
        })
    return pd.DataFrame(txns)

In [5]:
# ─────────────────────────────────────────
# TABLE 4: FRAUD LABELS
# ─────────────────────────────────────────
def generate_fraud_labels(txns_df):
    n = len(txns_df)
    is_fraud = np.random.choice([1, 0], size=n, p=[FRAUD_RATE, 1-FRAUD_RATE])
    
    fraud_type = []
    for f in is_fraud:
        if f == 1:
            fraud_type.append(random.choice(FRAUD_TYPES))
        else:
            fraud_type.append(None)
    
    return pd.DataFrame({
        'txn_id'     : txns_df['txn_id'],
        'is_fraud'   : is_fraud,
        'fraud_type' : fraud_type
    })


In [6]:
# ─────────────────────────────────────────
# TABLE 5: PAYMENT FAILURES
# ─────────────────────────────────────────
def generate_payment_failures(txns_df):
    # Only failed transactions get a record here
    failed_txns = txns_df[txns_df['status'] == 'Failed'][['txn_id']].copy()
    failed_txns['failure_reason'] = [random.choice(FAILURE_REASONS) 
                                      for _ in range(len(failed_txns))]
    failed_txns['retry_count']    = np.random.randint(0, 4, size=len(failed_txns))
    failed_txns['resolved']       = np.random.choice([True, False], 
                                                      size=len(failed_txns), 
                                                      p=[0.4, 0.6])
    return failed_txns.reset_index(drop=True)

In [7]:
# ─────────────────────────────────────────
# GENERATE ALL TABLES
# ─────────────────────────────────────────
print("Generating users...")
users_df     = generate_users()

print("Generating merchants...")
merchants_df = generate_merchants()

print("Generating transactions...")
txns_df      = generate_transactions(users_df, merchants_df)

print("Generating fraud labels...")
fraud_df     = generate_fraud_labels(txns_df)

print("Generating payment failures...")
failures_df  = generate_payment_failures(txns_df)

Generating users...
Generating merchants...
Generating transactions...
Generating fraud labels...
Generating payment failures...


In [9]:
import os

# Create folders if they don't exist
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/cleaned', exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [10]:
# ─────────────────────────────────────────
# SAVE RAW CSVs
# ─────────────────────────────────────────
users_df.to_csv('data/raw/users.csv', index=False)
merchants_df.to_csv('data/raw/merchants.csv', index=False)
txns_df.to_csv('data/raw/transactions.csv', index=False)
fraud_df.to_csv('data/raw/fraud_labels.csv', index=False)
failures_df.to_csv('data/raw/payment_failures.csv', index=False)

print(f"\nDataset Summary:")
print(f"  Users        : {len(users_df):,}")
print(f"  Merchants    : {len(merchants_df):,}")
print(f"  Transactions : {len(txns_df):,}")
print(f"  Fraud txns   : {fraud_df['is_fraud'].sum():,} ({FRAUD_RATE*100:.1f}%)")
print(f"  Failures     : {len(failures_df):,}")
print("\nAll CSVs saved to data/raw/")


Dataset Summary:
  Users        : 5,000
  Merchants    : 500
  Transactions : 50,000
  Fraud txns   : 1,624 (3.2%)
  Failures     : 3,517

All CSVs saved to data/raw/
